## srtm dem processing  
including dem mosaic, downsampling and clipping.


In [8]:
import os
from glob import glob
import rasterio as rio
import geopandas as gpd
from rasterio.merge import merge 
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling 


In [9]:
path_vec_ygp = 'data/boundary/ygp_region.gpkg'
ygp_vec_gdf = gpd.read_file(path_vec_ygp)


### Mosaic

In [10]:
paths_dem_ls = glob('data/dem/tiles/*')
path_mosaic = 'data/dem/SRTMGL3.tif'
src_files_to_mosaic = []
for fp in paths_dem_ls:
    src = rio.open(fp)
    src_files_to_mosaic.append(src)
mosaic_arr, mosaic_trans = merge(src_files_to_mosaic)
mosaic_meta = src.meta.copy()
mosaic_meta.update({
    "height": mosaic_arr.shape[1],
    "width": mosaic_arr.shape[2],
    "transform": mosaic_trans
    })

# Write the mosaic raster to disk
with rio.open(path_mosaic, 'w', **mosaic_meta) as dest:
    dest.write(mosaic_arr)


### Downsampling

In [11]:
path_srtm = 'data/dem/SRTMGL3.tif'
path_srtm_dsample = 'data/dem/SRTMGL3_005deg.tif' 

target_res = 0.005  
with rio.open(path_srtm) as src:
    transform, width, height = calculate_default_transform(
        src.crs, src.crs, src.width, src.height, *src.bounds,
        resolution=target_res)
    dst_meta = src.meta.copy()
    dst_meta.update({
        'transform': transform,
        'width': width,
        'height': height
    })
    with rio.open(path_srtm_dsample, 'w', **dst_meta) as dst:
        reproject(
            source=rio.band(src, 1),
            destination=rio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=src.crs,
            resampling=Resampling.average    # DEM 用 bilinear 或 average
        )


#### clip to ygp subregions. 

In [12]:
path_dem = 'data/dem/SRTMGL3_005deg.tif'
path_dem_save = 'data/dem/SRTMGL3_005deg_ygp.tif'

with rio.open(path_dem) as src:
    if ygp_vec_gdf.crs != src.crs: 
        ygp_vec_gdf = ygp_vec_gdf.to_crs(src.crs)      
    for idx, row in ygp_vec_gdf.iterrows():
        geom = row.geometry
        clipped_arr, clipped_transform = mask(dataset=src, shapes = [geom], 
                                              crop=True, all_touched=True)
        meta = src.meta.copy()
        meta.update({
            "height": clipped_arr.shape[1],
            "width": clipped_arr.shape[2],
            "transform": clipped_transform})
        ## save to path
        if os.path.exists(path_dem_save): os.remove(path_dem_save)
        with rio.open(path_dem_save, "w", **meta) as dst:
            print(dst.bounds)
            dst.write(clipped_arr)
        print(f"saved to: {path_dem_save}")



BoundingBox(left=97.2145833332709, bottom=20.640416666672323, right=109.15458333327089, top=30.105416666672323)
saved to: data/dem/SRTMGL3_005deg_ygp.tif
